[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline.ipynb)

# 01. 뉴스 제목으로 주제 맞히기 — 텍스트 분류 기준선

**"베트남 경제 고성장 지속…2분기 GDP 6.71% 성장"이라는 제목만 보고 이 기사가 `경제`인지
`세계`인지 맞히는 문제**입니다. 신문사 편집자가 기사를 어느 면에 넣을지 정하는 일을 모델에게 시키는 셈입니다.

## 이 장을 배우는 이유

지금까지 이 저장소에서 다룬 데이터는 **숫자와 범주**였습니다
([tabular-ml-practice](https://github.com/karzit/temp/blob/master/notebooks/tabular-ml-practice/README.md)).
표 데이터는 `fare`, `age`처럼 컬럼 하나가 곧 [피처](https://github.com/karzit/temp/blob/master/glossary.md#feature) 하나였습니다.
**텍스트에는 그런 컬럼이 없습니다.** 입력은 제목 문자열 하나뿐이고, 그 안에서 피처를 직접 만들어내야 합니다.
이 노트북의 절반은 "글자를 어떻게 숫자로 바꾸는가"에 대한 이야기입니다.

나머지 절반은 **"성능이 올랐다고 말해도 되는가"** 입니다. 텍스트 분류는 손댈 곳이 많습니다.
정제할까, 단어로 자를까 글자로 자를까, 모델은 무엇으로 할까. 하나씩 바꿔보면 숫자가 조금씩 움직이는데,
그중 어떤 것이 진짜 개선이고 어떤 것이 그냥 흔들림인지 구분하지 못하면 **아무 데나 시간을 쓰게 됩니다.**

## 이 노트북의 구성

| 절 | 내용 | 왜 하는가 |
|---|---|---|
| 1~3 | 데이터 관찰, **라벨을 누가 붙였는지** 확인 | 이 문제의 **상한선**을 먼저 가늠 |
| 4 | 기준선 만들기 | 나온 숫자가 잘한 건지 판단할 기준 |
| 5 | 텍스트를 숫자로 — BoW와 [TF-IDF](https://github.com/karzit/temp/blob/master/glossary.md#tfidf) | 텍스트 분류의 핵심 |
| 6~8 | 첫 모델, 전처리 실험, 문자 n-gram | 성능을 올리는 방법과 **올리지 못하는 방법** |
| 9~11 | 모델 비교, 평가 지표와 혼동 행렬, 오분류 분석 | 정확도 숫자 하나로 끝내지 않기 |
| 12 | 최종 성능을 재는 법 | 검증 점수를 그대로 믿으면 안 되는 이유 |

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행**하세요. 뒤 셀은 앞 셀에서 만든 변수를 씁니다
- 본문에 적힌 숫자는 **여러분이 실행한 결과와 소수점 이하가 다를 수 있습니다**
- 낯선 용어는 [glossary.md](https://github.com/karzit/temp/blob/master/glossary.md)에서 찾아보세요
- 에러가 나면 [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)를 보세요
- **소요 시간 50분쯤.** 오래 걸리는 셀은 8절(벡터화 비교)로, 셋을 합쳐 40초 안팎입니다

## 막혔을 때 — 이 노트북에서 자주 나오는 증상

| 증상 | 원인 | 대응 |
|---|---|---|
| 첫 셀에서 다운로드가 멈춤 | 46MB짜리 파일을 내려받는 중 | 30초쯤 기다려보고, 안 되면 셀을 다시 실행 |
| 그래프의 한글이 네모(□)로 | 한글 폰트 없음 | `koreanize-matplotlib` 설치 셀 실행 후 **런타임 재시작** |
| `ValueError: empty vocabulary` | 전처리가 텍스트를 전부 지움 | 정제 함수가 한글까지 지우지 않는지 확인 |

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

if IN_COLAB:
    !pip install -q pandas scikit-learn matplotlib seaborn koreanize-matplotlib

# KLUE-YNAT — 연합뉴스 기사 제목에 주제 라벨을 붙인 공개 데이터셋입니다.
# 파일을 미리 준비할 필요 없이, 아래 주소에서 그때그때 내려받아 씁니다.
YNAT = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/ynat-v1.1"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

try:
    import koreanize_matplotlib  # noqa: F401  (import만 해도 한글 폰트가 잡힙니다)
except ImportError:
    print("koreanize-matplotlib이 없어 그래프의 한글이 깨질 수 있습니다.")

pd.set_option("display.max_colwidth", 50)
RANDOM_STATE = 42

---

## 1. 어떤 데이터인가

**[KLUE-YNAT](https://klue-benchmark.com/tasks/66/overview/description)** 를 씁니다.
연합뉴스 기사 제목 45,678건에 사람이 주제 라벨을 붙인 공개 데이터셋이고,
한국어 자연어처리 모델을 평가하는 표준 벤치마크 KLUE의 일부입니다.

| 항목 | 내용 |
|---|---|
| 입력 | `title` — 기사 제목 (한 줄짜리 짧은 한국어) |
| 출력 | `label` — 주제 7개 중 하나 |
| 학습 데이터 | 45,678건 |
| 별도 평가 데이터 | 9,107건 — **12절에서 딱 한 번** 씁니다 |
| 출처·라이선스 | [KLUE-benchmark/KLUE](https://github.com/KLUE-benchmark/KLUE) · CC BY-SA 4.0 |

> **왜 이 데이터인가.** 짧고, 한국어이고, 라벨이 여러 개이고, **사람이 붙인 라벨이 얼마나 갈렸는지까지
> 기록되어 있습니다.** 마지막 항목이 특히 중요합니다. 대부분의 연습용 데이터는 정답이 하늘에서 떨어진 것처럼
> 주어지는데, 이 데이터에는 **주석자 세 명이 각각 무엇이라고 답했는지**가 남아 있습니다.
> 3절에서 이것을 확인합니다.

첫 다운로드는 46MB라 10~30초쯤 걸립니다. 제목과 라벨 말고도 여러 컬럼이 붙어 있어 파일이 큽니다.

In [ ]:
raw = pd.read_json(f"{YNAT}/ynat-v1.1_train.json")

print("전체:", raw.shape)
print("컬럼:", list(raw.columns))
raw[["title", "label"]].head()

컬럼이 일곱 개인데 우리가 쓸 것은 `title`과 `label` 둘뿐입니다. 나머지는 원본 기사 주소, 날짜,
그리고 3절에서 볼 주석 기록입니다.

**전체 45,678건 중 20,000건만 뽑아 씁니다.** 이유는 순전히 시간입니다. 전체를 쓰면 이 노트북의
모델 학습이 서너 배로 느려지는데, 배우는 내용은 달라지지 않습니다.
**데이터를 늘리면 성능이 얼마나 오르는지**는 연습 문제 5번에서 직접 확인합니다.

> `sample(random_state=...)`으로 뽑습니다. 앞에서부터 20,000건을 자르지 않는 이유는,
> 원본이 어떤 순서로 정렬되어 있는지 모르기 때문입니다. 날짜순이라면 앞부분만 잘라 쓸 때
> **특정 시기의 기사만** 보게 됩니다.

In [ ]:
N = 20_000
data = raw[["title", "label"]].sample(N, random_state=RANDOM_STATE).reset_index(drop=True)

print(len(data), "건")
data.head(8)

---

## 2. 데이터를 먼저 본다

모델을 짜기 전에 **데이터가 어떻게 생겼는지** 확인합니다. 표 데이터에서 `info()`/`describe()`로 했던 일을
텍스트에서도 똑같이 합니다. 볼 것은 세 가지입니다.

1. **결측과 중복** — 텍스트에서도 빈 값은 그대로 에러가 됩니다
2. **클래스 분포** — 몇 대 몇으로 치우쳐 있는가
3. **길이** — 제목이 몇 글자, 몇 단어짜리인가 (02번에서 시퀀스 길이를 정할 때 씁니다)

In [ ]:
print("결측:", data["title"].isna().sum(), "건")
print("완전 중복 행:", data.duplicated().sum(), "건")

# 같은 제목에 다른 라벨이 붙은 경우 — 있다면 라벨 오류의 직접적인 증거입니다.
충돌 = data.groupby("title")["label"].nunique()
print("같은 제목에 다른 라벨:", (충돌 > 1).sum(), "종")

**셋 다 0건입니다.** 잘 정리된 벤치마크 데이터라서 그렇습니다.

여기서 "그럼 이 점검은 헛수고였나" 하고 넘어가면 안 됩니다. **0건이라는 것을 확인한 것 자체가 결과입니다.**
확인하지 않고 지나갔다면, 나중에 성능이 이상할 때 데이터를 의심해야 할지 모델을 의심해야 할지
알 수 없습니다. 실무 데이터에서는 이 세 줄에서 대부분 뭔가가 걸립니다.

다음은 클래스 분포입니다.

In [ ]:
counts = data["label"].value_counts()
print(counts)
print("\n가장 많은 주제의 비율: %.3f" % (counts.iloc[0] / counts.sum()))

plt.figure(figsize=(8, 4))
sns.barplot(x=counts.values, y=counts.index, color="steelblue")
plt.title("주제별 건수")
plt.xlabel("건수")
plt.tight_layout()
plt.show()

In [ ]:
길이 = pd.DataFrame({
    "글자수": data["title"].str.len(),
    "단어수": data["title"].str.split().str.len(),
})
print(길이.describe().round(1))

**관찰 세 가지.**

- **클래스가 크게 치우치지는 않았습니다.** 가장 많은 `세계`가 18%, 가장 적은 `사회`가 11%입니다.
  표 데이터에서 자주 만나는 99:1 같은 극단적 불균형은 아닙니다. 그래도 균등(각 14.3%)은 아니므로,
  10절에서 평가 지표를 고를 때 이 사실을 다시 씁니다
- **제목은 짧습니다.** 평균 27글자, 6.6단어. 문단이 아니라 **한 줄**입니다
- 길이가 고르게 짧습니다(최대 44글자). 뒤에서 시퀀스 길이를 정할 때 편해집니다

여기까지가 흔히 하는 "데이터 품질 점검"입니다. **그런데 하나가 빠졌습니다. 정답 자체는 믿을 만한가?**

---

## 3. 라벨은 누가 붙였는가 — 이 문제의 상한선

`경제`인지 `IT과학`인지, `사회`인지 `정치`인지는 **사람이 봐도 갈리는** 판단입니다.
"네이버, AI 탑재 손목시계형 키즈폰 출시"는 IT 기사일까요, 경제 기사일까요?

보통은 여기서 추측만 하고 넘어갑니다. 이 데이터는 다릅니다. **주석자 세 명이 각각 무엇이라고 답했는지**가
`annotations` 컬럼에 남아 있습니다. 세 명이 얼마나 일치했는지 세어봅시다.

In [ ]:
# annotations 안의 'first-scope'가 주석자 세 명이 1순위로 고른 주제입니다.
first_scope = raw["annotations"].map(lambda a: a["annotations"]["first-scope"])

일치 = first_scope.map(lambda v: len(set(v)) == 1)
print("세 명이 모두 같은 답: %.1f%%" % (일치.mean() * 100))
print("2:1로 갈림       : %.1f%%" % ((~일치).mean() * 100))

# 갈린 예를 몇 개 봅니다.
갈린것 = raw.loc[~일치, ["title", "label"]].head(5)
for (제목, 정답), 표 in zip(갈린것.values, first_scope[~일치].head(5)):
    print(f"\n{제목}\n   주석자 셋: {표}  →  채택된 답: {정답}")

**세 명이 만장일치를 이룬 것은 64%뿐입니다. 나머지 36%는 2:1로 갈렸습니다.**

이 숫자가 뜻하는 바는 분명합니다. **정확도 100%는 목표가 아닙니다.** 사람 셋이 36%에서 이견을 냈다면,
그 경계선은 애초에 하나의 정답으로 딱 떨어지지 않습니다. 모델이 그 경계선을 전부 맞히려 든다면
그것은 규칙을 배운 것이 아니라 **채택된 답을 외운** 것입니다.

같은 이야기를 다른 각도에서도 볼 수 있습니다. 이 데이터에는 원래 기사가 실렸던 **언론사 지면 분류**도
함께 들어 있습니다(`predefined_news_category`). 주석자들이 다시 붙인 라벨과 비교해봅시다.

In [ ]:
불일치 = (raw["predefined_news_category"] != raw["label"])
print("언론사 지면과 주석자 라벨이 다른 비율: %.1f%%" % (불일치.mean() * 100))

pd.DataFrame({
    "언론사 지면": raw["predefined_news_category"].value_counts(),
    "주석자 라벨": raw["label"].value_counts(),
})

**17%가 다릅니다.** 표를 보면 어디서 어긋났는지 보입니다. **`사회`가 2,061건에서 5,133건으로 늘었습니다.**
주석자들은 `IT과학`이나 `정치`로 실렸던 기사 상당수를 `사회`로 다시 분류했습니다.
`사회`가 사실상 **"딱 떨어지지 않는 기사가 모이는 칸"** 역할을 한 것입니다.

이 사실을 기억해두세요. 10절에서 혼동 행렬을 볼 때 정확히 이 지점이 다시 나옵니다.

> **이것이 "데이터 품질 점검"의 다른 절반입니다.** 결측·중복을 세는 것은 쉽고 기계적입니다.
> 정작 성능의 천장을 정하는 것은 **라벨이 얼마나 잘 정의되어 있는가**인데, 그것은 세는 것만으로는
> 보이지 않습니다. 다행히 이 데이터는 주석 기록을 남겨줬습니다. 남겨주지 않는 데이터가 훨씬 많고,
> 그럴 때는 **직접 100건쯤 라벨을 붙여보고 원래 라벨과 얼마나 맞는지 재보는 것**이 같은 역할을 합니다.

---

## 4. 기준선 — 0.85는 잘한 걸까?

**모델 없이 얻을 수 있는 성능을 먼저 계산합니다.** 이 숫자를 넘지 못하면 그 모델은 존재할 이유가 없습니다.
가장 단순한 전략은 **무조건 가장 많은 주제로 찍기**입니다.

그전에 데이터를 둘로 나눕니다.

> **[학습/검증 분리](https://github.com/karzit/temp/blob/master/glossary.md#train-test-split)**: 가진 데이터의 일부(보통 20%)를
> 미리 떼어 숨겨두고, 학습에는 나머지만 씁니다. 그래야 "답을 외운 것"과 "규칙을 배운 것"을 구분할 수 있습니다.
> 학습에 쓴 데이터로 성능을 재면 시험 문제를 미리 보고 푼 것과 같습니다.

여기서 떼어둔 20%(`X_valid`)는 이 노트북 내내 **성능을 재는 자**로 씁니다.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

X = data["title"]
y = data["label"]

# stratify=y : 분할 후에도 주제 비율이 유지되도록 합니다. 특정 주제가 한쪽에만 몰리는 것을 막습니다.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("학습", len(X_train), "· 검증", len(X_valid))

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("무조건 최빈 주제로 찍기: 정확도 %.4f" % dummy.score(X_valid, y_valid))

**0.181입니다.** 주제가 7개니 무작위로 찍으면 0.143, 최빈값으로 찍으면 0.181.
앞으로 나올 숫자는 전부 이 기준선과 견줘서 읽으면 됩니다.

기준선을 계산하는 습관이 왜 중요한지는 반대 경우를 생각하면 분명합니다. 만약 어떤 주제가 전체의
80%였다면 **"정확도 78%"는 아무것도 안 한 것보다 나쁜 성적**입니다. 그런데도 78%라는 숫자만 보면
꽤 괜찮아 보입니다.

---

## 5. 텍스트를 숫자로 — BoW와 TF-IDF

머신러닝 모델은 문자열을 먹지 못합니다. `"남북정상 핫라인 열렸다"`를 숫자 배열로 바꿔야 합니다.
가장 기본적인 방법이 **BoW(Bag of Words, 단어 가방)** 입니다.

1. 전체 데이터에 나온 단어를 모아 **사전**을 만든다 (`남북정상`=0, `열렸다`=1, `핫라인`=2, ...)
2. 각 제목을 **사전 길이만큼의 벡터**로 바꾸고, 등장한 단어 자리에 횟수를 적는다

이름 그대로 **단어를 가방에 쓸어 담는** 방식이라 어순은 사라집니다. "북한이 미국을 비판"과
"미국이 북한을 비판"이 같은 벡터가 됩니다. 뜻이 정반대인데도 그렇습니다.
**그런데도 잘 통합니다.** 주제를 맞히는 데는 `누가 누구를`보다 `북한`·`미국`이라는 단어가 나왔다는
사실이 더 중요하기 때문입니다.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

예시 = [
    "남북정상 핫라인 열렸다",
    "북한날씨 흐리고 곳에 따라 눈",
    "마니커 140억원 제3자배정 유상증자 결정",
]

cv = CountVectorizer()
행렬 = cv.fit_transform(예시)

print("사전:", cv.get_feature_names_out())
print("\n행렬 크기:", 행렬.shape, "(문서 수 × 단어 수)")
pd.DataFrame(행렬.toarray(), columns=cv.get_feature_names_out(), index=["정치", "생활문화", "경제"])

> **읽는 법.** `fit_transform`이 돌려주는 것은 pandas 표가 아니라 **희소 행렬**(sparse matrix)입니다.
> 대부분의 칸이 0이라 0을 저장하지 않는 자료구조인데, 눈으로 보려면 `.toarray()`로 펼쳐야 합니다.
> 사전에 어떤 단어가 담겼는지는 `get_feature_names_out()`으로 확인합니다.
> 실제 데이터에서는 사전이 수만 개라 펼치면 화면을 덮으므로, **작은 예제에서만** 이렇게 봅니다.

**`140억원`이 `140억원`으로 남고 `제3자배정`도 통째로 남은 것을 보셨나요?** 반면 순수한 숫자만으로 된
토큰은 걸러집니다. `CountVectorizer`의 기본 토큰 규칙(`token_pattern`)이 **두 글자 이상의 낱말**만
남기기 때문입니다. 즉 **토큰화 기준이 곧 피처 설계**입니다.

한국어에는 더 큰 문제가 있습니다. **공백 기준으로 자르면 조사와 어미가 붙은 채로 잘립니다.**
`열렸다`, `열린`, `열립니다`가 전부 다른 단어가 됩니다. 제대로 하려면
[형태소 분석](https://github.com/karzit/temp/blob/master/glossary.md#morphological-analysis)(`kiwipiepy`, `konlpy` 등)이 필요합니다.
8절에서 형태소 분석기 없이 이 문제를 완화하는 방법(문자 n-gram)을 씁니다.
**그 방법이 이 노트북에서 가장 큰 성능 향상을 가져옵니다.**

### TF-IDF — 흔한 단어의 힘을 빼기

BoW에는 약점이 있습니다. `종합`, `속보`, `밝혀` 같은 말은 **모든 주제에 골고루** 나오는데,
등장 횟수만 세면 이런 단어도 큰 값을 갖습니다. [TF-IDF](https://github.com/karzit/temp/blob/master/glossary.md#tfidf)는
여기에 **"이 단어가 몇 개의 문서에 나오는가"** 로 벌점을 매깁니다.

```
TF-IDF(단어, 문서) = (문서 안 등장 횟수) × log(전체 문서 수 / 그 단어가 나온 문서 수)
```

**여러 문서에 두루 나오는 단어일수록 값이 작아집니다.** `손흥민`처럼 특정 주제에만 나오는 단어는
값이 커지고, `종합`처럼 아무 데나 붙는 단어는 값이 작아집니다. 사람이 "이 단어가 중요하다"고
지정하지 않아도 **데이터가 알아서** 정하는 셈입니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

예시2 = 예시 + ["코스피 상승 마감 종합", "손흥민 결승골 종합"]

tv = TfidfVectorizer()
tfidf = tv.fit_transform(예시2)

pd.DataFrame(tfidf.toarray(), columns=tv.get_feature_names_out()).round(2)

마지막 두 행을 보면 **두 문서에 모두 나온 `종합`의 값이, 같은 문서 안의 다른 단어보다 작습니다.**
반면 한 문서에만 나온 `손흥민`, `코스피`는 값이 큽니다. 이것이 TF-IDF가 하는 일의 전부입니다.

---

## 6. 첫 모델 — TF-IDF + 로지스틱 회귀

[로지스틱 회귀](https://github.com/karzit/temp/blob/master/glossary.md#logistic-regression)는 텍스트 분류의 **표준 출발점**입니다.
단어 수만 개짜리 희소 행렬에서 잘 동작하고, 빠르고, 어떤 단어가 어느 주제를 밀어올렸는지 볼 수 있습니다.

벡터화와 모델은 **`make_pipeline`으로 하나로 묶습니다.**

```
make_pipeline(TfidfVectorizer(), LogisticRegression())
   → fit(X, y)      : 벡터화를 fit_transform 하고, 그 결과로 모델을 fit
   → predict(새 X)  : 벡터화는 transform만 하고, 모델이 예측
```

이렇게 묶어두면 예측할 때 **전처리를 다시 재현할 필요가 없습니다.** 왜 이것이 중요한지는
바로 다음 셀 아래에서 설명합니다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
baseline.fit(X_train, y_train)

pred = baseline.predict(X_valid)
print("검증 정확도: %.4f" % accuracy_score(y_valid, pred))
print("사전 크기: %d 단어" % len(baseline.named_steps["tfidfvectorizer"].get_feature_names_out()))

**0.763.** 기준선 0.181에 비하면 큰 발전이지만, 아직 손댈 곳이 많습니다.

**사전 크기 40,429개를 눈여겨보세요.** 제목 16,000건에서 서로 다른 단어가 4만 개나 나왔습니다.
제목 하나가 6.6단어니 전체 토큰은 10만 개 남짓인데, 그중 4만 개가 서로 다릅니다.
**한국어에서 공백으로 자르면 이렇게 됩니다.** 이 숫자가 뒤에서 계속 발목을 잡습니다.

여기서 **`make_pipeline`을 쓴 이유**가 중요합니다. 벡터화와 모델을 하나로 묶으면
`fit`은 학습 데이터에만 적용되고, `predict` 때는 자동으로 `transform`만 호출됩니다.
직접 `TfidfVectorizer().fit_transform(전체_데이터)`를 부르면 **검증 데이터의 단어 분포가 사전에 섞여 들어가**
[데이터 누출](https://github.com/karzit/temp/blob/master/glossary.md#data-leakage)이 됩니다. 표 데이터에서 스케일러를
분할 전에 `fit`하면 안 됐던 것과 정확히 같은 문제입니다.

어떤 단어가 어느 주제를 밀어올렸는지도 볼 수 있습니다.

로지스틱 회귀는 단어마다 **계수(가중치)** 를 하나씩 가집니다. 계수가 크면 "이 단어가 나오면 이 주제일
가능성이 올라간다"는 뜻입니다. 모델의 계수는 `clf.coef_`에 `(주제 수 × 단어 수)` 모양으로 들어 있고,
`np.argsort`는 값을 정렬한 **순서(인덱스)** 를 돌려주므로 뒤에서 6개를 잘라내면 상위 6개 단어가 됩니다.

In [ ]:
import numpy as np

vec = baseline.named_steps["tfidfvectorizer"]
clf = baseline.named_steps["logisticregression"]
words = vec.get_feature_names_out()

for i, cat in enumerate(clf.classes_):
    top = np.argsort(clf.coef_[i])[-6:][::-1]
    print(f"{cat:<6}", ", ".join(words[j] for j in top))

**계수가 큰 단어가 상식과 맞는지 확인하는 습관을 들이세요.** 표 데이터에서 변수중요도로
데이터 누출을 잡아냈던 것과 같은 점검입니다.

여기서는 대체로 납득이 갑니다 — `코스피`·`특징주`가 경제, `감독`·`류현진`이 스포츠, `트럼프`·`이란`이 세계.
다만 눈에 걸리는 것도 있습니다. **`사회`의 상위 단어가 `kbs`, `방통위`, `mbc`, `게시판`입니다.**
사회 전반을 대표하는 말이라기보다 **언론사 관련 기사**에 가깝습니다.
3절에서 본 대로 `사회`가 "딱 떨어지지 않는 기사가 모이는 칸"이라, 모델이 붙잡을 만한
공통점이 마땅치 않았던 것입니다.

---

## 7. 전처리는 정말 도움이 되는가 — 그리고 "도움이 됐다"고 어떻게 말하는가

뉴스 제목에는 `…`, `·`, 숫자, 영문이 섞여 있고, 끝에 `종합`·`종합2보` 같은 **편집 표기**가 붙기도 합니다
(기사가 갱신될 때마다 붙는 표시입니다). "당연히 지우는 게 좋겠지"라고 생각하기 쉽습니다. **확인해봅시다.**

In [ ]:
import re


def clean_text(s):
    """제목에서 편집 표기·숫자·특수문자를 걷어낸다."""
    s = re.sub(r"종합\d*보?$", " ", s)      # 끝에 붙는 종합 / 종합2보
    s = re.sub(r"[0-9]+", " ", s)           # 숫자
    s = re.sub(r"[^가-힣A-Za-z ]", " ", s)  # 한글·영문·공백만 남기기
    return re.sub(r"\s+", " ", s).strip()   # 연속 공백 정리


for s in X_train.iloc[:4]:
    print(repr(s))
    print("   →", repr(clean_text(s)))

In [ ]:
X_train_c, X_valid_c = X_train.map(clean_text), X_valid.map(clean_text)

cleaned = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
).fit(X_train_c, y_train)

print("정제 전: %.4f" % accuracy_score(y_valid, baseline.predict(X_valid)))
print("정제 후: %.4f" % accuracy_score(y_valid, cleaned.predict(X_valid_c)))

**0.7632 → 0.7642. 0.001 올랐습니다.**

여기서 "정제가 도움이 된다"고 결론 내리고 싶어집니다. **그러면 안 됩니다.** 0.001은 검증 데이터
4,000건 중 **4건** 차이입니다. 분할을 다르게 했으면 반대로 나왔을 수도 있습니다.

그럼 어떻게 판단할까요. **분할을 여러 개 만들어서, 같은 분할 안에서 둘을 짝지어 비교합니다.**

In [ ]:
정제전, 정제후 = [], []

for seed in [0, 1, 2, 3, 4]:
    Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

    a = accuracy_score(yv, make_pipeline(
        TfidfVectorizer(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ).fit(Xt, yt).predict(Xv))

    b = accuracy_score(yv, make_pipeline(
        TfidfVectorizer(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ).fit(Xt.map(clean_text), yt).predict(Xv.map(clean_text)))

    정제전.append(a)
    정제후.append(b)
    print(f"분할 {seed}:  정제 전 {a:.4f}   정제 후 {b:.4f}   차이 {b - a:+.4f}")

정제전, 정제후 = np.array(정제전), np.array(정제후)
print(f"\n분할을 바꾸면 정확도가 {정제전.min():.4f} ~ {정제전.max():.4f} 사이에서 흔들립니다"
      f" (폭 {정제전.max() - 정제전.min():.4f})")
print(f"같은 분할 안에서 정제의 효과는 평균 {(정제후 - 정제전).mean():+.4f}"
      f" · 5번 중 {(정제후 > 정제전).sum()}번 정제 쪽이 이겼습니다")

**두 숫자를 나란히 놓고 보세요.**

| | 값 |
|---|---|
| 분할을 바꿀 때 정확도가 흔들리는 폭 | **0.0100** |
| 같은 분할에서 정제가 만든 차이 (평균) | **+0.0040** |

흔들림이 효과보다 **두 배 이상 큽니다.** 그러니 정확도 하나만 덜렁 보고 "0.7642가 0.7632보다 크다"고
말하는 것은 근거가 없습니다. **그런데도 정제가 도움이 된다고 말할 수 있습니다.** 5번 모두 정제 쪽이
이겼기 때문입니다.

**이 차이가 핵심입니다.**

- **절대 정확도**는 분할에 따라 크게 흔들립니다. 어떤 분할은 원래 쉽고, 어떤 분할은 어렵습니다
- **같은 분할 안에서의 차이**는 그 흔들림이 상쇄됩니다. 두 모델이 **똑같이 쉬운 문제**를 풀었기 때문입니다

이것을 **[짝지은 비교(paired comparison)](https://github.com/karzit/temp/blob/master/glossary.md#paired-comparison)** 라고 합니다. 두 방법을 비교할 때는 반드시
**같은 분할, 같은 seed**로 재고, **여러 분할에서 방향이 일관되는지** 보세요.
한 번 재서 나온 0.001은 아무것도 말해주지 않습니다.

**그래서 정제를 채택할까요?** 아직 결정하지 않습니다. 효과가 +0.004로 작기 때문입니다.
다음 절에서 **한 줄을 바꿔 +0.08을 얻습니다.** 그러고 나서 다시 판단합니다.

> **왜 정제의 효과가 이렇게 작을까요.** TF-IDF가 이미 비슷한 일을 하고 있기 때문입니다.
> `종합`은 모든 주제에 나오므로 IDF 가중치가 이미 낮습니다. 손으로 지워서 얻는 것이 크지 않습니다.
> 게다가 정제는 **정보를 지우는 작업**이라, 지운 것 중에 쓸모 있는 것이 있으면 손해입니다.

---

## 8. 문자 n-gram — 형태소 분석 없이 한국어 다루기

6절에서 사전이 40,429개나 됐던 것을 기억하세요. 원인은 **조사와 어미**입니다.
`정부는`, `정부가`, `정부의`, `정부와`가 전부 다른 단어로 세어집니다.
`열렸다`와 `열린`도 남남입니다. 사전은 커지는데 **각 단어가 나오는 횟수는 줄어들어**,
모델이 배울 것이 없어집니다.

**단어 대신 글자 몇 개씩 잘라서 세면** 이 문제가 크게 완화됩니다.

`analyzer="char_wb", ngram_range=(2, 3)`은 낱말 안에서 **글자 2~3개짜리 조각**을 만듭니다.

```
"정부는"  →  "정부", "부는", "정부는"
"정부가"  →  "정부", "부가", "정부가"
```

두 단어가 이제 `정부`라는 조각을 **공유합니다.** 조사가 무엇이든 어간은 살아남습니다.
대신 **피처 수가 크게 늘어납니다.**

아래 셀에서 세 가지를 비교하는데, 처음 보는 함수가 둘 나옵니다.

- **`make_union(A, B)`**: 벡터화 두 개를 나란히 돌려 **결과를 옆으로 이어 붙입니다.**
  단어 피처 4만 개 + 문자 피처 13만 개 = 17만 개짜리 입력이 됩니다. (`make_pipeline`이 **세로로**
  이어 붙이는 것이라면, `make_union`은 **가로로** 이어 붙이는 것입니다.)
- **`model[:-1]`**: 파이프라인에서 **마지막 단계(모델)를 뺀 앞부분**만 잘라낸 것입니다.
  여기에 `transform`을 부르면 "모델에 들어가기 직전의 숫자 배열"을 볼 수 있어, 피처가 몇 개인지 셀 수 있습니다.

**이 셀은 40초쯤 걸립니다.** 피처가 많아 학습이 느립니다.

In [ ]:
from sklearn.pipeline import make_union

configs = {
    "단어": TfidfVectorizer(),
    "문자 2~3": TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    "단어+문자": make_union(
        TfidfVectorizer(),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    ),
}

for name, vectorizer_ in configs.items():
    model = make_pipeline(vectorizer_, LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    model.fit(X_train, y_train)
    n_features = model[:-1].transform(X_train[:1]).shape[1]
    print(f"{name:<8} 정확도 {accuracy_score(y_valid, model.predict(X_valid)):.4f}  (피처 {n_features:,}개)")

**0.7632 → 0.8455. 8%p가 넘게 올랐습니다.**

앞 절에서 정제로 얻은 것이 +0.004였습니다. **여기서는 인자 두 개를 바꿔 +0.082를 얻었습니다.
스무 배입니다.** 이것이 7절에서 정제 채택을 미룬 이유입니다. 같은 시간을 쓴다면 어디에 써야 하는지가
분명해졌습니다.

**기억할 것:** 한국어 텍스트에서 형태소 분석기를 붙이기 전에 **문자 n-gram을 먼저 시도해보세요.**
설치도 필요 없고 인자 두 개면 됩니다. 형태소 분석기가 더 나은 경우도 있지만, 이 정도 효과를
공짜로 얻고 시작하는 편이 낫습니다.

**그런데 셋째 줄이 이상합니다.** 단어와 문자를 **둘 다** 쓴 `단어+문자`가 문자만 쓴 것보다
낮습니다(0.8405 < 0.8455). 정보를 더 줬는데 성능이 떨어졌습니다.

피처를 더한다고 좋아지지 않습니다. 단어 피처 4만 개는 대부분 **한두 번밖에 안 나오는 조각**이라,
모델이 그것을 외우는 데 가중치를 쓰면서 정작 쓸모 있는 문자 조각에 쓸 힘이 줄어듭니다.
**노이즈를 더한 셈입니다.**

이제 7절의 정제를 문자 n-gram 위에서 다시 재봅시다. 아까는 도움이 됐으니 여기서도 그럴까요?

In [ ]:
for 이름, 변환 in [("원본", lambda s: s), ("정제", clean_text)]:
    m = make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    ).fit(X_train.map(변환), y_train)
    print(f"{이름} %.4f" % accuracy_score(y_valid, m.predict(X_valid.map(변환))))

**정반대로 나왔습니다. 0.8455 → 0.8290, 1.7%p 손해입니다.**

| | 단어 단위 | 문자 단위 |
|---|---|---|
| 정제의 효과 | **+0.004** (도움) | **−0.017** (손해) |

같은 정제 함수인데 방향이 뒤집혔습니다. 이유는 이렇습니다. 문자 n-gram은 숫자와 기호까지
**조각으로 만들어 씁니다.** `6.71%`의 `%`, `2분기`의 `분기`, 날짜 표기 — 이런 것들이 경제 기사에
자주 붙는 신호였는데, 정제가 통째로 지워버렸습니다. 단어 단위에서는 어차피 `token_pattern`이
걸러내던 것들이라 지워도 잃을 게 없었습니다.

**전처리는 벡터화 방식과 짝을 이룹니다.** "이 전처리는 좋다/나쁘다"는 말은 성립하지 않고,
**"이 벡터화에서 이 전처리는 좋다/나쁘다"** 만 성립합니다. 앞 절에서 잰 +0.004는
단어 TF-IDF에 대한 이야기였을 뿐, 우리가 최종적으로 쓸 설정에는 해당하지 않았습니다.

**정제는 채택하지 않습니다.** 앞으로는 원본 텍스트에 문자 n-gram을 씁니다.

---

## 9. 모델 바꿔보기

벡터화 방식을 정했으니 분류 모델을 비교합니다. 텍스트에서 자주 쓰이는 세 가지입니다.

| 모델 | 성격 |
|---|---|
| `LogisticRegression` | 표준 출발점. 안정적이고 계수 해석이 가능 |
| `MultinomialNB` (나이브 베이즈) | 단어 등장 확률을 곱하는 고전적 방법. **아주 빠름** |
| `LinearSVC` | 고차원 희소 데이터에서 강한 선형 SVM |

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC


def vectorizer():
    """이 노트북에서 채택한 벡터화 설정."""
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))


models = {
    "로지스틱 회귀": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "나이브 베이즈": MultinomialNB(),
    "선형 SVM": LinearSVC(random_state=RANDOM_STATE),
}

fitted = {}
for name, model in models.items():
    pipe = make_pipeline(vectorizer(), model).fit(X_train, y_train)
    fitted[name] = pipe
    print(f"{name:<8} 정확도 {accuracy_score(y_valid, pipe.predict(X_valid)):.4f}")

**로지스틱 회귀 0.8455, 선형 SVM 0.8495, 나이브 베이즈 0.8090.**

7절에서 배운 기준으로 읽어봅시다. 로지스틱 회귀와 선형 SVM의 차이는 0.004로,
**분할을 바꾸면 순위가 뒤집힐 수 있는 폭**입니다. 둘 중 하나를 고르겠다면 짝지은 비교를 여러 분할에서
해봐야 합니다. 반면 나이브 베이즈와의 차이 0.04는 흔들림보다 훨씬 크므로 **실제로 뒤처지는 것**이 맞습니다.

그리고 앞 절과 견줘보세요. **벡터화를 바꿔서 +0.082, 모델을 바꿔서 +0.004입니다.**
텍스트 분류에서는 모델을 고르는 것보다 **입력을 어떻게 만드느냐가 대개 더 큰 차이**를 만듭니다.

이 노트북에서는 **로지스틱 회귀로 갑니다.** 성능이 사실상 같으면서
`predict_proba`로 예측 확률을 볼 수 있기 때문입니다(11절에서 씁니다). `LinearSVC`에는 그 기능이 없습니다.

---

## 10. 정확도 하나로 끝내면 안 되는 이유

정확도 0.85는 전체를 뭉뚱그린 숫자입니다. **어떤 주제는 잘 맞히고 어떤 주제는 못 맞히는지**가
그 안에 숨어 있습니다. 주제별로 나눠 봐야 합니다.

| 지표 | 뜻 | 언제 보나 |
|---|---|---|
| 정밀도(precision) | 이 주제라고 예측한 것 중 맞은 비율 | 잘못 넣는 비용이 클 때 |
| 재현율(recall) | 실제 이 주제 중 찾아낸 비율 | 놓치면 안 될 때 |
| f1-score | 둘의 조화평균 | 균형 |
| **macro avg** | **주제별 지표의 단순 평균** | **주제를 동등하게 볼 때** |
| weighted avg | 건수로 가중 평균한 값 | 정확도와 비슷하게 움직임 |

**macro 평균은 건수를 무시하고 주제를 동등하게 봅니다.** 건수가 적은 주제를 못 맞히면
정확도는 멀쩡해도 macro f1은 떨어집니다.

In [ ]:
from sklearn.metrics import classification_report, f1_score

best = fitted["로지스틱 회귀"]
pred = best.predict(X_valid)

print(classification_report(y_valid, pred, zero_division=0))
print("macro f1: %.4f" % f1_score(y_valid, pred, average="macro"))

**주제별 f1을 보세요. 스포츠 0.94에서 사회 0.62까지, 폭이 큽니다.**

특히 **`사회`의 재현율이 0.55입니다.** 실제 사회 기사의 **절반 가까이를 다른 주제로 보냈다**는 뜻입니다.
정확도 0.85만 봤다면 절대 알 수 없었을 사실입니다.

3절에서 예고한 지점이 여기입니다. `사회`는 주석자들이 **딱 떨어지지 않는 기사를 모아둔 칸**이었습니다.
공통된 단어가 없으니 모델이 붙잡을 것이 없습니다. 반대로 `스포츠`는 `감독`·`선수`·`경기`처럼
전용 어휘가 뚜렷해서 0.94가 나옵니다.

**어느 주제가 어느 주제로 갔는지**를 보면 더 분명해집니다. 그것을 한 장에 담은 표가
**혼동 행렬(confusion matrix)** 입니다.

> **읽는 법**: 행은 실제 주제, 열은 예측한 주제입니다. `(사회, 정치)` 칸의 숫자는
> "실제로는 사회인데 정치로 예측한 건수"입니다. **대각선은 맞힌 것**이고,
> 대각선 바깥이 전부 오답입니다.

In [ ]:
from sklearn.metrics import confusion_matrix

labels = sorted(y.unique())
cm = confusion_matrix(y_valid, pred, labels=labels)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("예측")
ax.set_ylabel("실제")
ax.set_title("혼동 행렬 — 어디서 헷갈리는가")
plt.tight_layout()
plt.show()

**혼동 행렬은 대각선이 아니라 대각선 바깥을 보는 그림입니다.**

`사회` 행을 가로로 훑어보세요. **오답이 특정 주제에 몰려 있지 않고 전부에 흩어져 있습니다** —
정치로 51건, 생활문화로 43건, IT과학으로 41건, 세계로 31건, 경제로 29건.
이것이 "잔여 범주"의 모습입니다. 만약 `사회`가 한 주제하고만 헷갈렸다면 그 둘의 경계를
다듬어서 고칠 수 있습니다. 전방위로 흩어진다면 **고칠 방법이 마땅치 않습니다.**

반면 고칠 여지가 있어 보이는 쌍도 있습니다.

- **경제 → IT과학 42건**: 통신사·반도체 기사가 양쪽에 걸칩니다
- **세계 → 정치 26건**: 외교 기사가 양쪽에 걸칩니다

**헷갈리는 쌍이 데이터의 성격 때문인지, 모델의 부족 때문인지**를 구분하는 것이 다음 단계입니다.
그러려면 틀린 것을 직접 봐야 합니다.

---

## 11. 틀린 것을 직접 본다

**성능을 올리는 가장 확실한 방법은 오분류를 눈으로 읽는 것입니다.** 숫자만 보면
"85%구나"에서 끝나지만, 틀린 제목 열몇 개를 읽으면 다음에 뭘 해야 할지 알게 됩니다.

이때 함께 보면 좋은 것이 **확신도**입니다. `predict`가 주제 하나만 돌려주는 것과 달리,
**`predict_proba`는 7개 주제 각각에 대한 확률**을 돌려줍니다(합이 1). 그중 최댓값이
"모델이 이 답에 얼마나 확신하는가"입니다. 0.95면 거의 확신한 것이고, 0.3이면 사실상 찍은 것입니다.

**확신도가 높은데 틀린 것부터** 봅니다. 모델이 자신 있게 틀리는 자리가 가장 많은 것을 알려줍니다.

In [ ]:
proba = best.predict_proba(X_valid)

오답 = pd.DataFrame({
    "제목": X_valid.values,
    "실제": y_valid.values,
    "예측": pred,
    "확신도": proba.max(axis=1).round(2),
})
오답 = 오답[오답["실제"] != 오답["예측"]].sort_values("확신도", ascending=False)

print(f"검증 {len(X_valid)}건 중 오답 {len(오답)}건")
오답.head(12)

**읽어보면 대부분 "모델이 틀렸다"고 말하기 어렵습니다.**

- `네이버 AI 탑재 손목시계형 키즈폰 아키 출시` — 정답은 `경제`, 모델은 `IT과학`
- `KT 28GHz 기업전용 5G 네트워크 개발 성공` — 정답은 `경제`, 모델은 `IT과학`
- `트럼프 유조선 공격 사소한 일…이란과 충돌우려 속 수위조절` — 정답은 `정치`, 모델은 `세계`

**여러분이라면 어느 쪽으로 분류하시겠습니까?** 통신사의 5G 개발 기사는 산업 기사이기도 하고
기술 기사이기도 합니다. 3절에서 본 **주석자 36% 불일치**가 정확히 이 자리에서 나타나고 있습니다.

오답을 세 종류로 나눠보면 이렇습니다.

| 유형 | 예 | 고칠 수 있나 |
|---|---|---|
| **경계가 애매한 기사** | 통신사 5G, 외교 관련 기사 | ✗ — 사람도 갈립니다 |
| **잔여 범주로 간 기사** | 실제 `사회`인데 다른 주제로 예측 | △ — `사회`의 정의를 바꾸지 않는 한 어렵습니다 |
| **제목만으로 알 수 없는 것** | `미세먼지 속 출근길` (3절에 나왔던 제목) | ✗ — 본문을 봐야 합니다 |

**세 유형 모두 모델을 키워서 해결되지 않습니다.** 남은 오답의 성격이 이렇다면, 다음에 할 일은
하이퍼파라미터 탐색이 아니라 **입력을 늘리는 것**(제목 대신 본문까지)이거나
**라벨 체계를 손보는 것**입니다.

확신도가 낮은 쪽은 어떨까요? 모델이 스스로 "모르겠다"고 말한 예측이 실제로 더 많이 틀리는지 봅니다.

In [ ]:
확신도 = proba.max(axis=1)
낮은순 = np.argsort(확신도)
하위400, 나머지 = 낮은순[:400], 낮은순[400:]

print("전체            정확도 %.4f" % accuracy_score(y_valid, pred))
print("확신도 하위 400건 정확도 %.4f  (확신도 %.2f 이하)"
      % (accuracy_score(y_valid.values[하위400], pred[하위400]), 확신도[하위400].max()))
print("나머지 3,600건   정확도 %.4f" % accuracy_score(y_valid.values[나머지], pred[나머지]))

**확신도 하위 10%의 정확도는 0.48, 나머지는 0.89입니다.**

모델이 내놓는 확률은 **자기가 얼마나 못 미더운지를 꽤 정직하게 알려줍니다.** 이 성질은 실무에서
그대로 쓰입니다. 확신도 낮은 10%만 사람이 검토하게 하면, **전체의 10%만 보고도 오답의 상당수를
걸러낼 수 있습니다.** 자동 분류 시스템이 대개 이런 식으로 만들어집니다 —
전부 자동으로 처리하는 대신, **애매한 것만 사람에게 넘깁니다.**

---

## 12. 이 숫자를 최종 성능이라고 말해도 되는가

지금까지 나온 검증 정확도가 0.8455입니다. 이것을 "이 모델의 성능"이라고 보고해도 될까요.

**안 됩니다.** 이 노트북에서 우리가 검증 세트를 본 횟수를 세어보세요. 정제할지 말지, 단어로 자를지
글자로 자를지, 모델을 무엇으로 할지 — **열 번 넘게 검증 세트 점수를 보고 결정했습니다.**
그때마다 검증 세트에 조금씩 맞춰간 셈입니다. 검증 세트는 더 이상 "본 적 없는 데이터"가 아닙니다.

**그래서 데이터를 셋으로 나눕니다.**

| 세트 | 쓰임 | 몇 번 보나 |
|---|---|---|
| 학습(train) | 모델을 학습 | 계속 |
| 검증(validation) | 방법을 고르는 데 사용 | 계속 |
| **테스트(test)** | **최종 성능 보고** | **딱 한 번** |

처음부터 이렇게 나눠뒀어야 합니다. 지금이라도 다시 나눠서, 지금까지의 선택
(문자 n-gram + 로지스틱 회귀)이 새 데이터에서도 통하는지 확인합니다.

In [ ]:
# 먼저 테스트 20%를 떼어 봉인하고, 남은 80%를 다시 학습/검증으로 나눕니다.
X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_tr2, X_va2, y_tr2, y_va2 = train_test_split(
    X_rest, y_rest, test_size=0.2, stratify=y_rest, random_state=RANDOM_STATE
)
print("학습 %d · 검증 %d · 테스트 %d" % (len(X_tr2), len(X_va2), len(X_test)))

m = make_pipeline(vectorizer(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
m.fit(X_tr2, y_tr2)
print("검증  %.4f" % accuracy_score(y_va2, m.predict(X_va2)))
print("테스트 %.4f" % accuracy_score(y_test, m.predict(X_test)))

**검증 0.8328, 테스트 0.8417.** 두 숫자가 가깝습니다. 방법 선택이 검증 세트에만 통하는 요령이
아니었다는 뜻입니다.

(검증 쪽이 오히려 조금 낮은데, 이 모델은 학습 데이터가 12,800건뿐이라 앞의 16,000건짜리보다
불리합니다. 성능을 재는 일이 끝났으니 **떼어뒀던 검증 데이터도 학습에 넣어** 최종 모델을 만듭니다.)

In [ ]:
final_model = make_pipeline(
    vectorizer(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
).fit(X_rest, y_rest)   # 학습 + 검증 = 16,000건

print("최종 모델 테스트 정확도: %.4f" % accuracy_score(y_test, final_model.predict(X_test)))

**0.8455.** 이 숫자가 우리가 보고할 성능입니다.

여기서 끝내도 되지만, 한 가지를 더 해봅니다. **이 모델을 진짜 새 데이터에 넣으면 어떻게 될까요?**

KLUE는 학습용과 별개로 **평가용 데이터 9,107건**을 따로 배포합니다. 우리 모델이 한 번도 본 적 없고,
우리가 나눈 것도 아닌 데이터입니다.

In [ ]:
dev = pd.read_json(f"{YNAT}/ynat-v1.1_dev.json")[["title", "label"]]
dev_pred = final_model.predict(dev["title"])

print("평가용 데이터 %d건" % len(dev))
print("정확도  %.4f" % accuracy_score(dev["label"], dev_pred))
print("macro f1 %.4f" % f1_score(dev["label"], dev_pred, average="macro"))

**0.8455에서 0.7639로, 8%p가 떨어졌습니다.**

테스트 세트에서는 잘 나왔는데 왜 여기서는 무너질까요. 모델이 잘못된 것이 아닙니다.
**데이터가 다릅니다.** 확인해봅시다.

In [ ]:
비교 = pd.DataFrame({
    "학습 데이터": raw["label"].value_counts(normalize=True),
    "평가 데이터": dev["label"].value_counts(normalize=True),
}).round(3)
비교["배수"] = (비교["평가 데이터"] / 비교["학습 데이터"]).round(1)
비교.sort_values("배수", ascending=False)

**`사회`가 학습 데이터에서는 11%인데 평가 데이터에서는 41%입니다. 3.6배입니다.**
반대로 `스포츠`는 17%에서 6%로 줄었습니다.

이제 8%p 하락이 설명됩니다. 우리 모델은 **`사회`를 가장 못 맞히는 모델**이었습니다(10절, 재현율 0.55).
그런데 새 데이터는 그 `사회`가 전체의 41%입니다. **모델의 가장 약한 부분이 정확히 가장 많이 나오는
데이터를 만난 것**입니다.

이것을 **[분포 이동(distribution shift)](https://github.com/karzit/temp/blob/master/glossary.md#distribution-shift)** 이라고 합니다. 실무에서 "테스트에서는 잘 나왔는데 배포하니
성능이 안 나온다"는 말의 대부분이 이것입니다. 모델은 그대로인데 **들어오는 데이터가 달라졌습니다.**

**여기서 배울 것 두 가지.**

1. **테스트 세트 점수는 "같은 분포에서 뽑았을 때"의 성능입니다.** 그 이상을 보장하지 않습니다.
   같은 데이터를 무작위로 쪼개서 잰 숫자는 아무리 정직하게 재도 이 한계를 넘지 못합니다
2. **약한 부분을 알고 있어야 합니다.** 10절에서 `사회` 재현율이 0.55라는 것을 확인해뒀기 때문에,
   지금 이 하락을 30초 만에 설명할 수 있었습니다. 정확도 하나만 봤다면 원인을 찾느라 한참 헤맸을 것입니다

> **실무에서는 어떻게 하나요.** 배포 후에도 **들어오는 데이터의 분포를 계속 감시합니다.**
> 라벨이 없어도 클래스별 예측 비율이 학습 때와 크게 달라지면 경고를 띄우는 식입니다.
> 그리고 새 분포의 데이터를 모아 다시 학습합니다.

---

## 정리

- **텍스트 분류는 "글자를 어떻게 숫자로 바꾸는가"가 절반**입니다. BoW → TF-IDF가 기본 경로입니다
- **TF-IDF는 흔한 단어의 힘을 자동으로 뺍니다.** 불용어 목록을 손으로 만들 필요가 크지 않습니다
- **기준선을 먼저 계산하세요.** 최빈 주제 0.181을 알아야 0.85가 어느 정도인지 판단됩니다
- **벡터화와 모델은 `Pipeline`으로 묶습니다.** 사전을 전체 데이터로 만들면 데이터 누출입니다
- **한 번 재서 나온 0.001은 아무것도 말해주지 않습니다.** 분할을 바꾸면 정확도가 0.01씩 흔들립니다.
  두 방법을 비교할 때는 **같은 분할에서 짝지어**, 여러 분할에서 방향이 일관되는지 보세요
- **한국어에서는 문자 n-gram(`char_wb`)을 먼저 시도**하세요. 이 데이터에서 **+8%p**로,
  이 노트북에서 가장 큰 향상이었습니다. 조사·어미 때문에 폭증한 사전 문제를 그대로 흡수합니다
- **전처리의 좋고 나쁨은 벡터화 방식에 달려 있습니다.** 같은 정제 함수가 단어 단위에서는 +0.004,
  문자 단위에서는 −0.017이었습니다
- **정확도 하나로 끝내지 마세요.** 전체 0.85 안에 f1 0.94짜리 주제와 0.62짜리 주제가 함께 있었습니다
- **오분류를 눈으로 읽으세요.** 남은 오답이 "사람도 갈리는 경계"라면, 다음 할 일은 모델 튜닝이 아닙니다
- **최종 성능은 딱 한 번 보는 테스트 세트에서 잽니다.** 그리고 그 숫자도
  **같은 분포에서만** 유효합니다 — 분포가 바뀌면 8%p가 그냥 날아갑니다

## 스스로 확인해보기

- [ ] BoW와 TF-IDF의 차이를 한 문장으로 설명할 수 있다
- [ ] `TfidfVectorizer`를 전체 데이터에 `fit`하면 왜 문제인지 안다
- [ ] 정확도 차이 0.001을 보고 "좋아졌다"고 말하면 안 되는 이유를 안다
- [ ] `analyzer="char_wb"`가 한국어에서 도움이 되는 이유를 설명할 수 있다
- [ ] 혼동 행렬에서 "잔여 범주"가 어떤 모습으로 나타나는지 읽을 수 있다
- [ ] 검증 세트와 테스트 세트를 따로 두는 이유를 설명할 수 있다
- [ ] 분포 이동이 무엇이고 왜 성능을 떨어뜨리는지 안다

## 연습 문제

풀어본 뒤 [01_text_baseline_solutions.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline_solutions.ipynb)에서 확인하세요.

**문제 1.** `사회`를 제외한 6개 주제만으로 학습하고 평가하면 정확도가 얼마나 오르나요?
그것을 "모델을 개선했다"고 말할 수 있을까요? 이 실험이 알려주는 것은 무엇인가요?

**문제 2.** `TfidfVectorizer`의 `min_df`(너무 드문 조각 제거)와 `max_df`(너무 흔한 조각 제거)를
바꿔가며 검증 정확도와 피처 수를 비교하세요. `min_df=1, 2, 5`와 `max_df=1.0, 0.5`의 조합으로 충분합니다.
피처를 줄여도 성능이 유지된다면 무엇이 좋은가요?

**문제 3.** `LogisticRegression(class_weight="balanced")`로 학습해 `사회`의 재현율이
어떻게 바뀌는지 `classification_report`로 비교하세요. 전체 정확도와 macro f1은 어떻게 되나요?
어느 쪽을 골라야 할까요?

**문제 4.** `GridSearchCV`로 벡터화 설정(`ngram_range`)과 로지스틱 회귀의 `C`를 함께 탐색하세요.
`Pipeline`의 파라미터 이름은 `단계이름__파라미터`(예: `tfidfvectorizer__ngram_range`) 형식입니다.
탐색 점수(`best_score_`)를 최종 성능으로 보고해도 될까요?
(문자 n-gram은 피처가 많아 탐색이 느립니다. **문제 2에서 손실이 없음을 확인한 `min_df=5`를
고정해두고** 시작하세요.)

**문제 5.** 학습 데이터를 5,000 / 20,000 / 45,678건(전체)으로 늘려가며 검증 정확도를 재보세요.
데이터를 늘려서 얻는 이득과, 8절에서 문자 n-gram으로 얻은 +8%p 중 어느 쪽이 큰가요?
(전체 데이터는 학습에 1분쯤 걸립니다.)

**문제 6.** 제목의 **앞 3단어만** 남긴 것과 **뒤 3단어만** 남긴 것으로 각각 학습해 비교하세요.
주제를 결정하는 정보가 제목의 앞쪽에 있나요, 뒤쪽에 있나요? 이 결과는 02번에서
시퀀스 길이를 정할 때 쓰입니다.

---

다음 노트북([02_keras_text](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text.ipynb))에서는
같은 문제를 Keras 신경망으로 풉니다. **딥러닝이 이 문제에서 TF-IDF를 이기는지** 확인하고,
지지 않는다면 **왜 지는지**를 숫자로 밝힙니다.